# 🧱 Day 3 Lab: Inspecting the Groundhog Basement

Welcome to Day 3. Today, we leave simple telemetry behind and dive deep into the Catalyst Optimizer's engine room. We will run `EXPLAIN FORMATTED` commands, token-scan physical plan outputs, and contrast distributed table-driven plans against tiny localized inline optimizations.

### Roadmap:
* **1. Execution Plan Dissection:** Mining an `EXPLAIN` string to locate recursive physical operators.
* **2. Token Verification:** Automating a scan over the plan text to map loop boundaries.
* **3. Tiny Town Optimizations:** Looking at how Spark executes light, tableless scalar counter math.
* **4. Inline Token Scanner:** Profiling the structural shift of an memory-only `LocalRelation` path.

### Initialize your Spark session (Ensure Spark 4.1.0+ or DBR 17.0+)

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

### CASE 1: Printing the Physical Hierarchy Plan
Let's re-register our corporate hierarchy data and output a full formatted query explanation.

In [ ]:
employees_data = [
    ("CEO_001", "Rita Root", None),
    ("VP_001",  "Victor VP", "CEO_001"),
    ("VP_002",  "Valerie VP", "CEO_001"),
    ("DIR_001", "Dina Director", "VP_001"),
    ("MGR_001", "Marco Manager", "DIR_001"),
    ("ENG_001", "Eddie Engineer", "MGR_001"),
    ("ENG_002", "Eva Engineer", "MGR_001")
]
spark.createDataFrame(employees_data, ["employee_id", "employee_name", "manager_id"]) \
     .createOrReplaceTempView("employees")

print("\n--- Case 1: Formatted Execution Plan for Tree Traversal ---")
plan_df = spark.sql("""
EXPLAIN FORMATTED
WITH RECURSIVE reports AS (
  SELECT employee_id, manager_id, 0 AS depth, array(employee_id) AS path
  FROM employees WHERE employee_id = 'CEO_001'
  UNION ALL
  SELECT e.employee_id, e.manager_id, r.depth + 1, concat(r.path, array(e.employee_id))
  FROM employees e
  JOIN reports r ON e.manager_id = r.employee_id
  WHERE r.depth < 20 AND NOT array_contains(r.path, e.employee_id)
)
SELECT * FROM reports
""")
plan_df.show(truncate=False)

### CASE 2: Structural Token Scanner
Let's write a quick automation function to verify if specific recursive relational tokens exist within our compiled plan text.

In [ ]:
print("\n--- Case 2: Verification of Internal Engine Tokens ---")
plan_text = "\n".join(row[0] for row in plan_df.collect())

target_tokens = [
    "UnionLoop",
    "UnionLoopRef",
    "Join",
    "Exchange",
    "LocalRelation",
    "OneRowRelation"
]

for token in target_tokens:
    is_present = token in plan_text
    status_emoji = "✅" if is_present else "❌"
    print(f"{status_emoji} Token '{token}' present in plan: {is_present}")

### CASE 3: Tiny Town Optimizations (The Counter Plan)
Let's run `EXPLAIN FORMATTED` on our simple numeric range counter. When executing loops over hardcoded values or scalars, Spark bypasses distributed cluster scheduling and condenses everything into single-threaded local relations.

In [ ]:
print("\n--- Case 3: Formatted Execution Plan for Inline Counter ---")
tiny_counter_plan = spark.sql("""
EXPLAIN FORMATTED
WITH RECURSIVE groundhog_day(n) AS (
  VALUES (1)
  UNION ALL
  SELECT n + 1 FROM groundhog_day WHERE n < 5
)
SELECT * FROM groundhog_day
""")
tiny_counter_plan.show(truncate=False)

### CASE 4: Token Scan Over Tiny Counter Plan
Let's token scan the counter plan to observe the structural shift toward `LocalRelation` optimization.

In [ ]:
print("\n--- Case 4: Token Scan Over Inline Counter ---")
tiny_plan_text = "\n".join(row[0] for row in tiny_counter_plan.collect())

for token in target_tokens:
    is_present = token in tiny_plan_text
    status_emoji = "✅" if is_present else "❌"
    print(f"{status_emoji} Token '{token}' present in tiny plan: {is_present}")

## 📊 Post-Lab Analysis: The Infrastructure Split

We have successfully dissected the internal representations of recursive operations in Catalyst. By contrasting the plans, we see two entirely different execution models masking behind the exact same syntax.

### 🔍 The Production Reality (Case 1 & 2)
Our production table query produced a heavy distributed physical layout featuring explicit `Join` and `Exchange` (Shuffle) operators. Spark does not walk pointers here. Every iteration requires checking intermediate frontiers, tracking path arrays, and spinning up independent task boundaries to perform raw dataframe joins against the base table.

### 📝 The "Tiny Town" Illusion (Case 3 & 4)
Our scalar counter plan dropped shuffles and structural joins entirely. The output shows a clean, inline layout:
```text
== Physical Plan ==
UnionLoop (1)
   :- LocalRelation (2)
   +- Project (5)
      +- Filter (4)
         +- UnionLoopRef (3)
```
* **`LocalRelation (2)`**: Because the data starts from a literal context (`VALUES (1)`), Spark writes the row data directly into the plan like a local memory variable. No table scans, no cluster scheduling.
* **Zero Shuffle Overhead**: The loop behaves like a native Python `while` loop running inside a single container driver. It finishes in under a second because it isn't actually distributing workloads.